# ADEA - Proiect empiric (faza I)
Autor: Dumitrescu Serban-Andrei

Equity: Wynn Resorts, Limited (NASDAQ: WYNN)

Esantion: Daily adjusted-close prices, 2005-01-03 to 2026-04-29 aproximativ 21 de ani

Conventie Returns : **Log Returns**

In acest notebook voi trece prin toate sectiunile, fiecare in parte avand o parte de computatie si o parte de discutie.

**Importam Datele si diverse**

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats

# diverse plots
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# datele(trebuie facut load ul inainte sau clonat repo)
df = pd.read_csv("../data/wynn_daily.csv", index_col=0, parse_dates=True)
log_returns = df["log_return"]

print(f"Loaded {len(log_returns)} daily log-return observations")
print(f"Range: {log_returns.index.min().date()} -> {log_returns.index.max().date()}")

Loaded 5363 daily log-return observations
Range: 2005-01-04 -> 2026-04-29


Momentele centrale

In [5]:
r = log_returns.values
n=len(r)

#momentele centrale

mean=log_returns.mean()
var= log_returns.var()
std=log_returns.std()
skew=log_returns.skew()
exkurt=log_returns.kurtosis()

results= pd.DataFrame({"Tabel_Momente":[mean,var,std,skew,exkurt]},index=["mean","variance","std dev","skewness","excess kurtosis"])

print(results.to_string(float_format=lambda x: f"{x:.8f}"))



                 Tabel_Momente
mean                0.00020446
variance            0.00098889
std dev             0.03144666
skewness            0.16032423
excess kurtosis     8.20629831


Anualizam

In [6]:
annual_mean= mean*252
annual_vol=std*np.sqrt(252)

print(f"Daily mean:        {mean:.6f}  ({mean*100:.4f}%)")
print(f"Annualized mean:   {annual_mean:.4f}  ({annual_mean*100:.2f}%)")
print(f"Daily std dev:     {std:.6f}  ({std*100:.4f}%)")
print(f"Annualized vol:    {annual_vol:.4f}  ({annual_vol*100:.2f}%)")
print()
print(f"Skewness:          {skew:+.4f}   (Normal: 0)")
print(f"Excess kurtosis:   {exkurt:+.4f}   (Normal: 0)")

Daily mean:        0.000204  (0.0204%)
Annualized mean:   0.0515  (5.15%)
Daily std dev:     0.031447  (3.1447%)
Annualized vol:    0.4992  (49.92%)

Skewness:          +0.1603   (Normal: 0)
Excess kurtosis:   +8.2063   (Normal: 0)


Eliminam outliers ca sa vedem cat de sensibile sunt skew si kurt la ele

In [10]:
lower, upper = np.percentile(r, [1, 99])
r_trimmed = r[(r >= lower) & (r <= upper)]

print(f"Original sample:  n = {len(r)},   "
      f"skew = {stats.skew(r):+.4f},   exkurt = {stats.kurtosis(r):+.4f}")
print(f"Trimmed at 1/99%: n = {len(r_trimmed)},  "
      f"skew = {stats.skew(r_trimmed):+.4f},   exkurt = {stats.kurtosis(r_trimmed):+.4f}")
print()
print("Scapand de doar 2 la suta din date osbervam ca momentele superioare se schimba dramatic, in special kurtosis")

Original sample:  n = 5363,   skew = +0.1603,   exkurt = +8.1975
Trimmed at 1/99%: n = 5255,  skew = +0.1643,   exkurt = +1.0388

Scapand de doar 2 la suta din date osbervam ca momentele superioare se schimba dramatic, in special kurtosis


 **Comentarii 1.1**

Observam ca **media zilnica** este pozitiva dar f mica (de asteptat) la 0.02%, ceea ce corespunde anualizat unei medii de ~5.2%. Pe o perioada de aproape 21 de ani WYNN a generat o rentabilitate cu adevarat modesta, insa si asta cumva nu ma mira avand in vedere sectorul in care activeaza ( Turism, jocuri de noroc in Macau)

**Volatilitatea** zilnica este de aproape 3.2% ceea ce anualizat conduce la un staggering 50% (idem ca mai sus)

**Skewness** este usor pozitiv. (+0.16). aceasta valoare poate parea un pic neobisnuita pentru rentabilitati pe actiuni , insa pare robusta, deoarece dupa ce am eliminat 1% val extreme a ramas aproape neschimbat.

**Kurtosis** de 8.20 este observatia cea mai importanta a acestei actiuni. Pentru distributia normala, excess kurtosis = 0; o valoare de 8 indică cozi *dramatic*
mai grele decât cele ale normalei și este motivul fundamental al respingerii ipotezei
de normalitate (vezi 1.5)

Spre deosebire de skewness, **kurtosis-ul este extrem de fragil la valori extreme**:
trunchierea a doar 2% din observații (1% pe fiecare coadă) reduce excess kurtosis-ul
de la 8,20 la 1,04 — o scădere de 8x. AStfel  observam că non-normalitatea
distribuției WYNN este concentrată aproape exclusiv în cozi, în timp ce corpul central al distribuției este aproape gaussian.

Ce mi se pare extrem de interesant este ca sub ipoteza de normalitate Valoarea minima inregistrata la 16 martie 2020 (in plin lockdown de covid) de -28.02% ar avea sub o normala $\mathcal{N}(\hat\mu, \hat\sigma^2)$ probabilitatea
$\Phi(-0,2802 / 0,0314) = \Phi(-8,92) \approx 10^{-19}$ , adica infinitezimal de mica. intr o lume gaussiana nu ar trebui sa apara in niciun univers vizibil de zile de tranzactionare. si totusi s-a intamplat :) E o ilustrare directa a limitelor asumptiei de normalitate in datele financiare.


